# Modeling Objective

This notebook compares baseline, traditional time-series, and machine-learning approaches for forecasting total daily sales for store `CA_1` over a 28-day horizon. The target is aggregated store-level daily unit sales, not item-level sales.

## Load and Prepare Data

Load the transformed Parquet dataset, aggregate all item sales by date, and confirm the daily series is chronological and complete.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid", context="notebook")

PRIMARY_COLOR = "#2F6F9F"
SECONDARY_COLOR = "#D98E04"
ACCENT_COLOR = "#5E8C61"

def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "processed" / "ca1_sales_long.parquet").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing data/processed/ca1_sales_long.parquet")

PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "ca1_sales_long.parquet"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"
FORECASTS_DIR = PROJECT_ROOT / "outputs" / "forecasts"

def save_figure(fig: plt.Figure, filename: str) -> Path:
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    output_path = FIGURES_DIR / filename
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return output_path

In [ ]:
raw_df = pd.read_parquet(DATA_PATH)

if not pd.api.types.is_datetime64_any_dtype(raw_df["date"]):
    raw_df["date"] = pd.to_datetime(raw_df["date"])

raw_df["event_indicator"] = raw_df[["event_name_1", "event_name_2"]].notna().any(axis=1).astype(int)

daily_sales = (
    raw_df.groupby("date", as_index=False)
    .agg(
        sales=("sales", "sum"),
        event_indicator=("event_indicator", "max"),
        snap_CA=("snap_CA", "max"),
    )
    .sort_values("date")
    .reset_index(drop=True)
)

expected_dates = pd.date_range(daily_sales["date"].min(), daily_sales["date"].max(), freq="D")
missing_dates = expected_dates.difference(daily_sales["date"])
duplicate_dates = daily_sales["date"].duplicated().sum()

data_summary = pd.DataFrame(
    [
        {"metric": "Minimum date", "value": daily_sales["date"].min()},
        {"metric": "Maximum date", "value": daily_sales["date"].max()},
        {"metric": "Observations", "value": daily_sales.shape[0]},
        {"metric": "Missing dates", "value": len(missing_dates)},
        {"metric": "Duplicate dates", "value": int(duplicate_dates)},
    ]
)

display(data_summary)
display(daily_sales.head())

if len(missing_dates) > 0 or duplicate_dates > 0:
    raise ValueError("Daily sales series must have no missing or duplicate dates before modeling.")

## Time-Based Validation

Use rolling validation folds with a 28-day forecast horizon. Random train-test splits are not used because validation dates must always occur after training dates.

In [ ]:
HORIZON = 28
N_FOLDS = 3

def create_rolling_folds(data: pd.DataFrame, horizon: int = 28, n_folds: int = 3) -> list[dict]:
    if len(data) < horizon * (n_folds + 1):
        n_folds = max(1, len(data) // horizon - 1)
    folds = []
    for fold_number in range(n_folds, 0, -1):
        val_start_idx = len(data) - horizon * fold_number
        val_end_idx = val_start_idx + horizon
        train = data.iloc[:val_start_idx].copy()
        valid = data.iloc[val_start_idx:val_end_idx].copy()
        folds.append({"fold": len(folds) + 1, "train": train, "valid": valid})
    return folds

folds = create_rolling_folds(daily_sales, HORIZON, N_FOLDS)

fold_summary = pd.DataFrame(
    [
        {
            "fold": fold["fold"],
            "train_start": fold["train"]["date"].min(),
            "train_end": fold["train"]["date"].max(),
            "validation_start": fold["valid"]["date"].min(),
            "validation_end": fold["valid"]["date"].max(),
            "train_days": fold["train"].shape[0],
            "validation_days": fold["valid"].shape[0],
            "validation_after_training": fold["valid"]["date"].min() > fold["train"]["date"].max(),
        }
        for fold in folds
    ]
)

fold_summary

## Evaluation Metrics

Evaluate each model with MAE, RMSE, and RMSSE. The RMSSE denominator is calculated only from each fold's training data.

In [ ]:
def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return mean_absolute_error(y_true, y_pred)

def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return mean_squared_error(y_true, y_pred) ** 0.5

def rmsse_denominator(train_values: pd.Series, seasonality: int = 1) -> float:
    diffs = train_values.diff(seasonality).dropna()
    scale = np.mean(np.square(diffs))
    return scale if scale > 0 else np.nan

def rmsse(y_true: np.ndarray, y_pred: np.ndarray, train_values: pd.Series, seasonality: int = 1) -> float:
    denominator = rmsse_denominator(train_values, seasonality)
    if pd.isna(denominator):
        return np.nan
    return np.sqrt(np.mean(np.square(y_true - y_pred)) / denominator)

def evaluate_forecast(model_name: str, fold_number: int, train: pd.DataFrame, valid: pd.DataFrame, predictions: np.ndarray) -> dict:
    y_true = valid["sales"].to_numpy()
    y_pred = np.clip(np.asarray(predictions, dtype=float), 0, None)
    return {
        "Model": model_name,
        "Fold": fold_number,
        "MAE": mae(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "RMSSE": rmsse(y_true, y_pred, train["sales"], seasonality=1),
    }

## Baseline Models

Compare three simple baselines: last observed value, weekly seasonal naive, and 28-day seasonal naive.

In [ ]:
def naive_forecast(train: pd.DataFrame, horizon: int) -> np.ndarray:
    return np.repeat(train["sales"].iloc[-1], horizon)

def seasonal_naive_forecast(train: pd.DataFrame, horizon: int, seasonality: int) -> np.ndarray:
    history = train["sales"].to_numpy()
    return np.array([history[-seasonality + (step % seasonality)] for step in range(horizon)])

baseline_models = {
    "Naive": lambda train, horizon: naive_forecast(train, horizon),
    "Seasonal Naive 7": lambda train, horizon: seasonal_naive_forecast(train, horizon, 7),
    "Seasonal Naive 28": lambda train, horizon: seasonal_naive_forecast(train, horizon, 28),
}

## Traditional Time-Series Model

Use Holt-Winters Exponential Smoothing with additive trend and weekly seasonality. Forecasts are true 28-day multi-step forecasts and do not use validation sales.

In [ ]:
def holt_winters_forecast(train: pd.DataFrame, horizon: int) -> np.ndarray:
    series = train.set_index("date")["sales"].asfreq("D")
    model = ExponentialSmoothing(
        series,
        trend="add",
        seasonal="add",
        seasonal_periods=7,
        initialization_method="estimated",
    )
    fitted = model.fit(optimized=True)
    return fitted.forecast(horizon).to_numpy()

## Feature Engineering for Machine Learning

Create historical lag, shifted rolling, and calendar features. Rolling features are shifted by one day before calculation to prevent future-data leakage.

In [ ]:
FEATURE_COLUMNS = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_28",
    "day_of_week",
    "month",
    "year",
    "is_weekend",
    "event_indicator",
    "snap_CA",
]

def add_calendar_features(data: pd.DataFrame) -> pd.DataFrame:
    result = data.copy()
    result["day_of_week"] = result["date"].dt.dayofweek
    result["month"] = result["date"].dt.month
    result["year"] = result["date"].dt.year
    result["is_weekend"] = result["day_of_week"].isin([5, 6]).astype(int)
    return result

def make_supervised_features(data: pd.DataFrame) -> pd.DataFrame:
    result = add_calendar_features(data).sort_values("date").reset_index(drop=True)
    shifted_sales = result["sales"].shift(1)
    for lag in [1, 7, 14, 28]:
        result[f"lag_{lag}"] = result["sales"].shift(lag)
    for window in [7, 28]:
        result[f"rolling_mean_{window}"] = shifted_sales.rolling(window).mean()
        result[f"rolling_std_{window}"] = shifted_sales.rolling(window).std()
    return result.dropna(subset=FEATURE_COLUMNS + ["sales"]).reset_index(drop=True)

feature_preview = make_supervised_features(daily_sales).head()
feature_preview

## Machine-Learning Model

Train a Random Forest on dates before each validation period. Forecast validation days recursively by appending each prediction to the history before creating the next day's lag and rolling features.

In [ ]:
def make_random_forest() -> RandomForestRegressor:
    return RandomForestRegressor(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=1,
    )

def feature_row_from_history(history: list[float], calendar_row: pd.Series) -> dict:
    row = {
        "lag_1": history[-1],
        "lag_7": history[-7],
        "lag_14": history[-14],
        "lag_28": history[-28],
        "rolling_mean_7": float(np.mean(history[-7:])),
        "rolling_mean_28": float(np.mean(history[-28:])),
        "rolling_std_7": float(np.std(history[-7:], ddof=1)),
        "rolling_std_28": float(np.std(history[-28:], ddof=1)),
        "day_of_week": calendar_row["date"].dayofweek,
        "month": calendar_row["date"].month,
        "year": calendar_row["date"].year,
        "is_weekend": int(calendar_row["date"].dayofweek in [5, 6]),
        "event_indicator": int(calendar_row["event_indicator"]),
        "snap_CA": int(calendar_row["snap_CA"]),
    }
    return row

def random_forest_recursive_forecast(train: pd.DataFrame, future_calendar: pd.DataFrame, horizon: int) -> tuple[np.ndarray, RandomForestRegressor]:
    train_features = make_supervised_features(train)
    model = make_random_forest()
    model.fit(train_features[FEATURE_COLUMNS], train_features["sales"])

    history = train["sales"].astype(float).tolist()
    predictions = []
    for _, calendar_row in future_calendar.head(horizon).iterrows():
        feature_row = pd.DataFrame([feature_row_from_history(history, calendar_row)], columns=FEATURE_COLUMNS)
        prediction = max(float(model.predict(feature_row)[0]), 0.0)
        predictions.append(prediction)
        history.append(prediction)
    return np.array(predictions), model

## Model Comparison

Evaluate all models on identical rolling validation periods and identify the best model using the lowest average RMSSE.

In [ ]:
metric_rows = []
prediction_frames = []
rf_models = {}

for fold in folds:
    fold_number = fold["fold"]
    train = fold["train"]
    valid = fold["valid"]
    horizon = valid.shape[0]

    for model_name, forecast_function in baseline_models.items():
        predictions = forecast_function(train, horizon)
        metric_rows.append(evaluate_forecast(model_name, fold_number, train, valid, predictions))
        prediction_frames.append(valid[["date", "sales"]].assign(Model=model_name, Fold=fold_number, predicted_sales=np.clip(predictions, 0, None)))

    hw_predictions = holt_winters_forecast(train, horizon)
    metric_rows.append(evaluate_forecast("Holt-Winters", fold_number, train, valid, hw_predictions))
    prediction_frames.append(valid[["date", "sales"]].assign(Model="Holt-Winters", Fold=fold_number, predicted_sales=np.clip(hw_predictions, 0, None)))

    rf_predictions, rf_model = random_forest_recursive_forecast(train, valid[["date", "event_indicator", "snap_CA"]], horizon)
    rf_models[fold_number] = rf_model
    metric_rows.append(evaluate_forecast("Random Forest", fold_number, train, valid, rf_predictions))
    prediction_frames.append(valid[["date", "sales"]].assign(Model="Random Forest", Fold=fold_number, predicted_sales=rf_predictions))

metrics_table = pd.DataFrame(metric_rows).sort_values(["Fold", "RMSSE"]).reset_index(drop=True)
predictions_table = pd.concat(prediction_frames, ignore_index=True)
predictions_table["residual"] = predictions_table["sales"] - predictions_table["predicted_sales"]

summary_table = (
    metrics_table.groupby("Model", as_index=False)
    .agg(MAE=("MAE", "mean"), RMSE=("RMSE", "mean"), RMSSE=("RMSSE", "mean"))
    .sort_values("RMSSE")
    .reset_index(drop=True)
)

best_model_name = summary_table.iloc[0]["Model"]
strongest_baseline = summary_table[summary_table["Model"].isin(baseline_models.keys())].iloc[0]
best_model_summary = summary_table.iloc[0]

METRICS_DIR.mkdir(parents=True, exist_ok=True)
metrics_output_path = METRICS_DIR / "model_comparison.csv"
summary_output_path = METRICS_DIR / "model_comparison_summary.csv"
metrics_table.to_csv(metrics_output_path, index=False)
summary_table.to_csv(summary_output_path, index=False)

display(metrics_table)
display(summary_table)
print(f"Best model by average RMSSE: {best_model_name}.")
print(f"Fold-level metrics saved to: {metrics_output_path}")
print(f"Summary metrics saved to: {summary_output_path}")

## Visual Evaluation

Plot final-fold predictions, average RMSSE by model, and residuals for the best model.

In [ ]:
final_fold = max(predictions_table["Fold"])
final_fold_predictions = predictions_table[predictions_table["Fold"].eq(final_fold)].copy()

fig, ax = plt.subplots(figsize=(12, 5))
actual_final = final_fold_predictions.drop_duplicates("date")[["date", "sales"]]
ax.plot(actual_final["date"], actual_final["sales"], label="Actual", color="black", linewidth=2)
for model_name, model_data in final_fold_predictions.groupby("Model"):
    ax.plot(model_data["date"], model_data["predicted_sales"], label=model_name, linewidth=1.5)
ax.set_title("Actual vs Predicted Sales, Final Validation Fold")
ax.set_xlabel("Date")
ax.set_ylabel("Total daily sales")
ax.legend()
fig.autofmt_xdate()
actual_vs_predicted_path = save_figure(fig, "model_actual_vs_predicted_final_fold.png")

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=summary_table, x="Model", y="RMSSE", color=PRIMARY_COLOR, ax=ax)
ax.set_title("Average RMSSE by Model")
ax.set_xlabel("Model")
ax.set_ylabel("Average RMSSE")
ax.tick_params(axis="x", rotation=30)
rmsse_comparison_path = save_figure(fig, "model_rmsse_comparison.png")

best_residuals = predictions_table[predictions_table["Model"].eq(best_model_name)].copy()
fig, ax = plt.subplots(figsize=(12, 5))
ax.axhline(0, color="black", linewidth=1)
ax.scatter(best_residuals["date"], best_residuals["residual"], color=SECONDARY_COLOR, alpha=0.8)
ax.set_title(f"Residuals for Best Model: {best_model_name}")
ax.set_xlabel("Date")
ax.set_ylabel("Actual - predicted sales")
fig.autofmt_xdate()
residual_plot_path = save_figure(fig, "model_best_residuals.png")

pd.DataFrame(
    [
        {"figure": "Actual vs predicted", "path": actual_vs_predicted_path},
        {"figure": "Average RMSSE", "path": rmsse_comparison_path},
        {"figure": "Best-model residuals", "path": residual_plot_path},
    ]
)

## Feature Importance

Train a Random Forest on all available supervised rows and inspect feature importances. These importances describe predictive contribution, not causal impact.

In [ ]:
rf_full_training = make_supervised_features(daily_sales)
rf_full_model = make_random_forest()
rf_full_model.fit(rf_full_training[FEATURE_COLUMNS], rf_full_training["sales"])

feature_importance = (
    pd.DataFrame({"feature": FEATURE_COLUMNS, "importance": rf_full_model.feature_importances_})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

top_feature_importance = feature_importance.head(10)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=top_feature_importance.sort_values("importance"), x="importance", y="feature", color=PRIMARY_COLOR, ax=ax)
ax.set_title("Random Forest Top Feature Importances")
ax.set_xlabel("Importance")
ax.set_ylabel("Feature")
feature_importance_path = save_figure(fig, "model_random_forest_feature_importance.png")

display(top_feature_importance)
print(f"Feature importance chart saved to: {feature_importance_path}")
print("The highest-importance features are the historical and calendar inputs the Random Forest used most for prediction. This does not prove causality.")

## Final 28-Day Forecast

Retrain the best-performing approach on all available CA_1 daily data and produce the next 28-day forecast where technically appropriate. Negative predictions are clipped to zero.

In [ ]:
def final_baseline_forecast(model_name: str, data: pd.DataFrame, horizon: int) -> np.ndarray:
    if model_name == "Naive":
        return naive_forecast(data, horizon)
    if model_name == "Seasonal Naive 7":
        return seasonal_naive_forecast(data, horizon, 7)
    if model_name == "Seasonal Naive 28":
        return seasonal_naive_forecast(data, horizon, 28)
    raise ValueError(f"Unsupported baseline model: {model_name}")

def forecast_with_best_available_model(best_model: str, data: pd.DataFrame, horizon: int) -> tuple[str, np.ndarray, str]:
    if best_model in baseline_models:
        return best_model, final_baseline_forecast(best_model, data, horizon), "Best validation model is directly forecastable."
    if best_model == "Holt-Winters":
        return best_model, holt_winters_forecast(data, horizon), "Best validation model is directly forecastable."
    if best_model == "Random Forest":
        non_exogenous_summary = summary_table[summary_table["Model"].ne("Random Forest")].sort_values("RMSSE")
        fallback_model = non_exogenous_summary.iloc[0]["Model"]
        note = (
            "Random Forest was best on validation, but the transformed dataset does not contain future event and SNAP values "
            "for the next 28 days. The final export uses the strongest directly forecastable non-exogenous model instead."
        )
        if fallback_model in baseline_models:
            return fallback_model, final_baseline_forecast(fallback_model, data, horizon), note
        return fallback_model, holt_winters_forecast(data, horizon), note
    raise ValueError(f"Unsupported best model: {best_model}")

forecast_model_name, final_predictions, final_forecast_note = forecast_with_best_available_model(best_model_name, daily_sales, HORIZON)
forecast_dates = pd.date_range(daily_sales["date"].max() + pd.Timedelta(days=1), periods=HORIZON, freq="D")

final_forecast = pd.DataFrame(
    {
        "date": forecast_dates,
        "predicted_sales": np.clip(final_predictions, 0, None),
        "model": forecast_model_name,
    }
)

FORECASTS_DIR.mkdir(parents=True, exist_ok=True)
forecast_output_path = FORECASTS_DIR / "ca1_28_day_forecast.csv"
final_forecast.to_csv(forecast_output_path, index=False)

display(final_forecast)
print(f"Forecast model used for final export: {forecast_model_name}.")
print(final_forecast_note)
print(f"Forecast saved to: {forecast_output_path}")

## Business Interpretation

Use the computed validation metrics and forecast outputs to summarize the most useful business takeaways.

In [ ]:
baseline_improvement = (
    (strongest_baseline["RMSSE"] - best_model_summary["RMSSE"]) / strongest_baseline["RMSSE"] * 100
    if strongest_baseline["RMSSE"] != 0
    else np.nan
)

business_interpretation = pd.DataFrame(
    [
        {"topic": "Best-performing model", "finding": f"{best_model_name} had the lowest average RMSSE across validation folds."},
        {"topic": "Improvement over strongest baseline", "finding": f"Average RMSSE improvement versus {strongest_baseline['Model']}: {baseline_improvement:.2f}%."},
        {"topic": "Important patterns", "finding": "Historical sales, weekly timing, calendar signals, events, and SNAP patterns are candidate drivers for later feature development."},
        {"topic": "Inventory and staffing", "finding": "The 28-day store-level forecast can support near-term inventory planning and staffing decisions for CA_1."},
        {"topic": "Limitations", "finding": "This notebook models only aggregated CA_1 sales, so it does not capture item-level or cross-store differences."},
        {"topic": "Recommendation", "finding": "Expand the pipeline to item and store levels after validating the aggregated CA_1 workflow."},
    ]
)

business_interpretation